# CNN 教學範例：Colab 自動下載 EMNIST Digits

教學重點：

1. 自動下載資料集
2. 建立 DataLoader
3. 建立簡單但實用的 CNN
4. 訓練與測試模型
5. 儲存與讀取模型
6. 模擬使用者提供一張圖片並預測結果

In [ ]:
import os  # 匯入 os，用來建立與管理 Colab 的資料夾路徑
import torch  # 匯入 PyTorch，負責張量運算與深度學習模型
import torchvision  # 匯入 Torchvision，負責影像資料集與影像工具
import torchvision.transforms as transforms  # 匯入 transforms，負責圖片前處理
from torch import nn  # 匯入 neural network 模組，用來建立 CNN
from torch.utils.data import DataLoader  # 匯入 DataLoader，用來分批讀取資料
from torch.utils.data import Subset  # 匯入 Subset，用來取部分資料加快教學訓練
import matplotlib.pyplot as plt  # 匯入 matplotlib，用來顯示圖片與訓練曲線
import numpy as np  # 匯入 NumPy，用來處理陣列資料
from sklearn.metrics import confusion_matrix  # 匯入 confusion_matrix，用來建立混淆矩陣
from sklearn.metrics import ConfusionMatrixDisplay  # 匯入 ConfusionMatrixDisplay，用來畫出混淆矩陣
from sklearn.metrics import classification_report  # 匯入 classification_report，用來顯示分類指標

## 1. 設定 GPU 與基本參數

In [ ]:
torch.manual_seed(0)  # 固定 PyTorch 隨機種子，讓訓練結果比較容易重現
np.random.seed(0)  # 固定 NumPy 隨機種子，讓資料處理結果比較容易重現

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # 如果 Colab 有 GPU 就使用 cuda，否則使用 CPU
print('使用裝置:', device)  # 顯示目前使用的是 GPU 還是 CPU

batch_size = 128  # 設定每次訓練讀取 128 張圖片
epochs = 5  # 設定訓練 5 輪，教學時先不要設太大
learning_rate = 0.001  # 設定 Adam optimizer 的學習率
num_classes = 10  # EMNIST digits 有 10 個類別，也就是數字 0 到 9

data_dir = '/content/emnist_data'  # 設定 EMNIST 資料集下載位置，Colab 建議放在 /content 底下
model_path = '/content/cnn_emnist_digits.pt'  # 設定模型儲存位置，也放在 /content 底下
os.makedirs(data_dir, exist_ok=True)  # 如果 /content/emnist_data 不存在，就自動建立資料夾

use_small_data = True  # 教學時先使用部分資料，加快訓練速度
train_size = 20000  # 設定教學用訓練資料筆數
test_size = 5000  # 設定教學用測試資料筆數


## 2. 設定圖片前處理

EMNIST 在 torchvision 裡常需要旋轉與左右翻轉，顯示方向才會比較正常。

In [ ]:
transform_train = transforms.Compose([  # 建立訓練資料的圖片前處理流程
    lambda img: transforms.functional.rotate(img, -90),  # 將 EMNIST 圖片順時針旋轉 90 度
    lambda img: transforms.functional.hflip(img),  # 將 EMNIST 圖片左右翻轉
    transforms.RandomRotation(10),  # 訓練時隨機旋轉小角度，增加模型泛化能力
    transforms.ToTensor(),  # 將 PIL 圖片轉成 PyTorch Tensor
    transforms.Normalize((0.5,), (0.5,))  # 將灰階像素標準化到大約 -1 到 1 的範圍
])  # 結束訓練資料前處理流程

transform_test = transforms.Compose([  # 建立測試資料的圖片前處理流程
    lambda img: transforms.functional.rotate(img, -90),  # 將 EMNIST 圖片順時針旋轉 90 度
    lambda img: transforms.functional.hflip(img),  # 將 EMNIST 圖片左右翻轉
    transforms.ToTensor(),  # 將 PIL 圖片轉成 PyTorch Tensor
    transforms.Normalize((0.5,), (0.5,))  # 將灰階像素標準化到大約 -1 到 1 的範圍
])  # 結束測試資料前處理流程


## 3. 自動下載 EMNIST Digits 資料集

In [ ]:
train_data = torchvision.datasets.EMNIST(root=data_dir, split='digits', train=True, download=True, transform=transform_train)  # 自動下載並建立訓練資料
test_data = torchvision.datasets.EMNIST(root=data_dir, split='digits', train=False, download=True, transform=transform_test)  # 自動下載並建立測試資料

if use_small_data == True:  # 如果是教學模式，就只取部分資料
    train_data = Subset(train_data, range(train_size))  # 取前 train_size 筆訓練資料
    test_data = Subset(test_data, range(test_size))  # 取前 test_size 筆測試資料

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)  # 建立訓練資料 DataLoader，並打亂資料順序
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)  # 建立測試資料 DataLoader，測試時不需要打亂資料

print('訓練資料筆數:', len(train_data))  # 顯示訓練資料筆數
print('測試資料筆數:', len(test_data))  # 顯示測試資料筆數

## 4. 顯示部分訓練圖片

In [ ]:
images, labels = next(iter(train_loader))  # 從訓練資料中取出一個 batch

grid = torchvision.utils.make_grid(images[:16], nrow=8, normalize=True)  # 將前 16 張圖片排成格子

plt.figure(figsize=(8, 4))  # 設定顯示圖片的大小
plt.imshow(grid.permute(1, 2, 0))  # 將 Tensor 維度轉成 matplotlib 可以顯示的格式
plt.title('Labels: ' + str(labels[:16].tolist()))  # 顯示前 16 張圖片的答案
plt.axis('off')  # 關閉座標軸
plt.show()  # 顯示圖片

## 5. 建立簡單 CNN 模型

模型設計：

- 兩層卷積層
- 每層卷積後接 BatchNorm、ReLU、MaxPool
- 最後接全連接層分類
- 加入 Dropout，降低過度擬合


![](https://i.urusai.cc/OUT3B.jpg)

![](https://i.urusai.cc/m2KgT.jpg)

In [ ]:
class SimpleCNN(nn.Module):  # 定義一個簡單 CNN 類別
    def __init__(self):  # 定義模型初始化方法
        super().__init__()  # 呼叫 nn.Module 的初始化方法

        self.features = nn.Sequential(  # 建立卷積特徵抽取區塊
            # 第 1 層卷積特徵抽取區塊
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),  # 第一層卷積，輸入 1 通道，輸出 32 通道
            nn.BatchNorm2d(num_features=32),  # 對 32 個通道 (32張特徵圖) 做 Batch Normalization，讓訓練更穩定
            nn.ReLU(),  # 使用 ReLU activation function
            nn.MaxPool2d(kernel_size=2),  # 使用 2x2 Max Pooling，將 28x28 變成 14x14

            # 第 2 層卷積特徵抽取區塊
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),  # 第二層卷積，輸入 32 通道，輸出 64 通道
            nn.BatchNorm2d(num_features=64),  # 對 64 個通道 (64張特徵圖) 做 Batch Normalization，讓訓練更穩定
            nn.ReLU(),  # 使用 ReLU activation function
            nn.MaxPool2d(kernel_size=2)  # 使用 2x2 Max Pooling，將 14x14 變成 7x7
        )  # 結束卷積特徵抽取區塊

        self.classifier = nn.Sequential(  # 建立分類器區塊
            nn.Flatten(),  # 將 64x7x7 的特徵圖攤平成一維向量
            nn.Dropout(p=0.25),  # 隨機關閉 25% 神經元，降低過度擬合
            nn.Linear(in_features=64 * 7 * 7, out_features=128),  # 建立全連接層，將特徵轉成 128 維
            nn.ReLU(),  # 使用 ReLU activation function
            nn.Linear(in_features=128, out_features=num_classes)  # 輸出 10 類的分數
        )  # 結束分類器區塊

    def forward(self, x):  # 定義模型的 forward propagation
        x = self.features(x)  # 先通過卷積特徵抽取區塊
        x = self.classifier(x)  # 再通過分類器區塊
        return x  # 回傳每一個類別的分數

model = SimpleCNN().to(device)  # 建立模型並放到 GPU 或 CPU
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)  # 計算可訓練參數數量
print(model)  # 顯示模型結構
print('可訓練參數數量:', total_params)  # 顯示模型可訓練參數數量

## 6. 設定 loss function 與 optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()  # 使用 CrossEntropyLoss 處理多類別分類問題
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)  # 使用 Adam 更新模型參數

## 7. 訓練 CNN 模型

In [ ]:
train_loss_history = []  # 建立 list，用來記錄每一個 epoch 的訓練 loss
train_acc_history = []  # 建立 list，用來記錄每一個 epoch 的訓練 accuracy
test_acc_history = []  # 建立 list，用來記錄每一個 epoch 的測試 accuracy

for epoch in range(epochs):  # 重複訓練 epochs 次
    model.train()  # 將模型切換成訓練模式
    total_loss = 0  # 建立變數，用來累加訓練 loss
    train_correct = 0  # 建立變數，用來累加訓練資料預測正確數
    train_total = 0  # 建立變數，用來累加訓練資料總數

    for images, labels in train_loader:  # 逐批讀取訓練圖片與答案
        images = images.to(device)  # 將圖片放到 GPU 或 CPU
        labels = labels.to(device)  # 將答案放到 GPU 或 CPU
        optimizer.zero_grad()  # 清空上一輪留下來的梯度
        outputs = model(images)  # 將圖片輸入模型並取得預測分數

        loss = criterion(outputs, labels)  # 計算預測分數與真實答案之間的 loss
        loss.backward()  # 使用 backpropagation 計算梯度
        optimizer.step()  # 根據梯度更新模型參數

        total_loss += loss.item() * images.size(0)  # 累加這個 batch 的總 loss
        preds = outputs.argmax(dim=1)  # 取出分數最高的類別當作預測結果
        train_correct += (preds == labels).sum().item()  # 累加預測正確的數量
        train_total += labels.size(0)  # 累加目前看過的訓練資料數量
        
    train_loss = total_loss / train_total  # 計算這個 epoch 的平均訓練 loss
    train_acc = train_correct / train_total  # 計算這個 epoch 的訓練 accuracy
    model.eval()  # 將模型切換成評估模式

    test_correct = 0  # 建立變數，用來累加測試資料預測正確數
    test_total = 0  # 建立變數，用來累加測試資料總數
    with torch.no_grad():  # 評估時不計算梯度，節省記憶體與時間
        for images, labels in test_loader:  # 逐批讀取測試圖片與答案
            images = images.to(device)  # 將圖片放到 GPU 或 CPU
            labels = labels.to(device)  # 將答案放到 GPU 或 CPU
            outputs = model(images)  # 將圖片輸入模型並取得預測分數
            preds = outputs.argmax(dim=1)  # 取出分數最高的類別當作預測結果
            test_correct += (preds == labels).sum().item()  # 累加測試資料預測正確數
            test_total += labels.size(0)  # 累加目前看過的測試資料數量
    
    test_acc = test_correct / test_total  # 計算這個 epoch 的測試 accuracy
    train_loss_history.append(train_loss)  # 記錄這個 epoch 的訓練 loss
    train_acc_history.append(train_acc)  # 記錄這個 epoch 的訓練 accuracy
    test_acc_history.append(test_acc)  # 記錄這個 epoch 的測試 accuracy
    print(f'Epoch {epoch + 1}/{epochs}, Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}')  # 印出這個 epoch 的結果

## 8. 畫出訓練過程

In [ ]:
plt.figure(figsize=(8, 4))  # 設定圖表大小
plt.plot(train_loss_history, label='Train Loss')  # 畫出每個 epoch 的訓練 loss
plt.xlabel('Epoch')  # 設定 X 軸名稱
plt.ylabel('Loss')  # 設定 Y 軸名稱
plt.title('Training Loss')  # 設定圖表標題
plt.legend()  # 顯示圖例
plt.show()  # 顯示圖表

plt.figure(figsize=(8, 4))  # 設定圖表大小
plt.plot(train_acc_history, label='Train Accuracy')  # 畫出每個 epoch 的訓練 accuracy
plt.plot(test_acc_history, label='Test Accuracy')  # 畫出每個 epoch 的測試 accuracy
plt.xlabel('Epoch')  # 設定 X 軸名稱
plt.ylabel('Accuracy')  # 設定 Y 軸名稱
plt.title('Accuracy')  # 設定圖表標題
plt.legend()  # 顯示圖例
plt.show()  # 顯示圖表

## 9. 儲存與讀取模型

In [ ]:
torch.save(model.state_dict(), model_path)  # 儲存模型參數到指定檔案
loaded_model = SimpleCNN().to(device)  # 建立一個新的 SimpleCNN 模型
loaded_model.load_state_dict(torch.load(model_path, map_location=device))  # 讀取剛才儲存的模型參數
loaded_model.eval()  # 將讀取後的模型切換成評估模式
print('模型已儲存並重新讀取:', model_path)  # 顯示模型儲存與讀取完成訊息


## 10. 使用測試資料評估模型

In [ ]:
all_preds = []  # 建立 list，用來儲存所有預測結果
all_labels = []  # 建立 list，用來儲存所有真實答案

loaded_model.eval()  # 將模型切換成評估模式

with torch.no_grad():  # 評估時不計算梯度，節省記憶體與時間
    for images, labels in test_loader:  # 逐批讀取測試圖片與答案
        images = images.to(device)  # 將圖片放到 GPU 或 CPU
        outputs = loaded_model(images)  # 使用讀取後的模型進行預測
        preds = outputs.argmax(dim=1)  # 取出分數最高的類別當作預測結果
        all_preds.append(preds.cpu().numpy())  # 將預測結果移回 CPU 並存入 list
        all_labels.append(labels.numpy())  # 將真實答案存入 list
all_preds = np.concatenate(all_preds)  # 將多個 batch 的預測結果合併成一個陣列
all_labels = np.concatenate(all_labels)  # 將多個 batch 的真實答案合併成一個陣列
target_names = [str(i) for i in range(num_classes)]  # 建立類別名稱，也就是字串形式的 0 到 9
print(classification_report(all_labels, all_preds, target_names=target_names))  # 顯示 precision、recall、f1-score

cm = confusion_matrix(all_labels, all_preds)  # 建立混淆矩陣
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)  # 建立混淆矩陣顯示物件
fig, ax = plt.subplots(figsize=(8, 8))  # 建立圖表與座標軸
disp.plot(ax=ax, cmap='Blues', values_format='d')  # 畫出混淆矩陣
plt.title('Confusion Matrix')  # 設定圖表標題
plt.show()  # 顯示圖表


## 11. 模擬使用者提供一張圖片並預測

實務上，使用者上傳圖片後，也要做和訓練資料相同的前處理，才能交給模型預測。這裡先用測試集中的一張圖片模擬。

In [ ]:
sample_image, true_label = test_data[0]  # 從測試資料取出一張圖片和真實答案
input_image = sample_image.unsqueeze(0).to(device)  # 增加 batch 維度，讓形狀從 1x28x28 變成 1x1x28x28

loaded_model.eval()  # 將模型切換成評估模式

with torch.no_grad():  # 預測時不計算梯度，節省記憶體與時間
    output = loaded_model(input_image)  # 將圖片輸入模型並取得每一類的分數
    probabilities = torch.softmax(output, dim=1)  # 將分數轉成每一類的機率
    pred_label = probabilities.argmax(dim=1).item()  # 取出機率最高的類別
    pred_prob = probabilities.max().item()  # 取出最高類別的機率

plt.figure(figsize=(3, 3))  # 設定圖片顯示大小
plt.imshow(sample_image.squeeze(), cmap='gray')  # 顯示灰階圖片
plt.title(f'True: {true_label}, Pred: {pred_label}')  # 顯示真實答案與預測答案
plt.axis('off')  # 關閉座標軸
plt.show()  # 顯示圖片

print('真實答案:', true_label)  # 印出真實答案
print('預測答案:', pred_label)  # 印出預測答案
print('預測機率:', pred_prob)  # 印出最高類別的預測機率


## 12. 自己上傳一張圖片並預測

In [ ]:
from google.colab import files  # 匯入 Colab 的檔案上傳工具
from PIL import Image, ImageOps  # 匯入 PIL，用來讀取與處理圖片
import numpy as np  # 匯入 numpy，用來判斷圖片亮度
import torch  # 匯入 PyTorch，用來建立 tensor 與模型預測
import torchvision.transforms as transforms  # 匯入 transforms，用來做圖片前處理
import matplotlib.pyplot as plt  # 匯入 matplotlib，用來顯示圖片

uploaded = files.upload()  # 讓使用者從電腦上傳一張圖片

image_path = list(uploaded.keys())[0]  # 取得使用者上傳的第一張圖片檔名
img = Image.open(image_path).convert('L')  # 讀取圖片，並轉成灰階圖片
img = ImageOps.autocontrast(img)  # 自動調整圖片對比，讓數字更明顯
img_array = np.array(img)  # 將圖片轉成 numpy array，方便判斷背景亮度
if img_array.mean() > 127:  # 如果平均亮度大於 127，通常代表背景偏白
    img = ImageOps.invert(img)  # 將圖片反相，變成黑底白字，接近 EMNIST 格式

img = img.resize((28, 28))  # 將圖片縮放成模型需要的 28x28 大小

upload_transform = transforms.Compose([  # 建立使用者上傳圖片的前處理流程
    transforms.ToTensor(),  # 將 PIL 圖片轉成 PyTorch Tensor，形狀會變成 1x28x28
    transforms.Normalize((0.5,), (0.5,))  # 使用和訓練資料相同的標準化方式
])  # 結束前處理流程

input_image = upload_transform(img)  # 對使用者圖片做前處理
input_image = input_image.unsqueeze(0)  # 增加 batch 維度，形狀從 1x28x28 變成 1x1x28x28
input_image = input_image.to(device)  # 將圖片放到 GPU 或 CPU

loaded_model.eval()  # 將模型切換成評估模式

with torch.no_grad():  # 預測時不需要計算梯度
    output = loaded_model(input_image)  # 將使用者圖片輸入模型，取得每個類別的分數
    probabilities = torch.softmax(output, dim=1)  # 將分數轉成每個數字的機率
    pred_label = probabilities.argmax(dim=1).item()  # 取得機率最高的數字類別
    pred_prob = probabilities.max().item()  # 取得最高的預測機率

plt.figure(figsize=(3, 3))  # 建立圖片顯示區域
plt.imshow(img, cmap='gray')  # 顯示處理後的 28x28 灰階圖片
plt.title(f'Pred: {pred_label}')  # 在圖片標題顯示預測結果
plt.axis('off')  # 關閉座標軸
plt.show()  # 顯示圖片

print('預測數字:', pred_label)  # 印出模型預測的數字
print('預測機率:', pred_prob)  # 印出模型對該數字的信心分數

## 13. 簡短觀察
- 訓練 accuracy 是否持續上升？
- 測試 accuracy 是否和訓練 accuracy 差很多？
- 哪些數字容易被模型搞混？
- 如果增加 epochs，模型是否變好？
- 如果拿掉 Dropout，是否更容易 overfitting？

如果要辨識動物，資料集該去哪裡下載？你可以用 AI 來協助你完成這些實作。